# Calibrated Stacking vs Strong Tabular Baselines for Credit Risk
Run cells in order (or Runtime > Run all). Each result is saved to Drive in `credit_risk_v2/out`, and finished steps are skipped on re-run. Use a GPU runtime for Taiwan.

In [ ]:
# 1) SETUP. Run this cell first (Runtime > Run all runs it first). Safe to re-run.
import os, sys, glob, subprocess, urllib.request
try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive'
except ImportError:                      # local test
    ROOT = os.environ.get('CR_ROOT', os.getcwd())

# find the folder that holds credit_benchmark.py (works wherever it sits in Drive)
hits = sorted(glob.glob(ROOT + '/credit_risk_v2/code/credit_benchmark.py')) or \
       sorted(glob.glob(ROOT + '/**/credit_risk_v2/code/credit_benchmark.py', recursive=True))
assert hits, 'credit_benchmark.py not found under credit_risk_v2/code in your Drive'
CODE = os.path.dirname(hits[0])
BASE = os.path.dirname(CODE)             # .../credit_risk_v2
OUT, DATA = BASE + '/out', BASE + '/data'
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
for f in ('credit_benchmark.py', 'analyze_benchmark.py', 'make_figures.py', 'verify_paper.py'):
    assert os.path.exists(f'{CODE}/{f}'), f'missing {f} in {CODE}'
os.environ['CR_BASE'] = BASE

# data from the public repo (only downloads what is missing)
RAW = 'https://raw.githubusercontent.com/RamGhadiyaram/calibrated-stacked-credit-risk/main/data/'
for f in ('GermanCredit.csv', 'taiwan_credit.csv'):
    if not os.path.exists(f'{DATA}/{f}') or os.path.getsize(f'{DATA}/{f}') < 1000:
        urllib.request.urlretrieve(RAW + f, f'{DATA}/{f}')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pytorch-tabnet', 'catboost', 'lightgbm',
                'xgboost', 'tabulate', 'matplotlib'], check=False)

def run(args):
    """Run a script from the code folder with CR_BASE set, streaming its output."""
    p = subprocess.Popen([sys.executable, '-u'] + args, cwd=CODE, env=dict(os.environ, CR_BASE=BASE),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if line.strip():
            print(line, end='')
    p.wait()
    print('exit code', p.returncode)
    return p.returncode

print('CODE', CODE); print('BASE', BASE); print('data', sorted(os.listdir(DATA))); print('out', sorted(os.listdir(OUT)))


In [ ]:
# 2) GERMAN: 3 repeats x 5 folds. Skipped automatically if already finished.
assert 'run' in globals(), 'Run cell 1 first'
if os.path.exists(OUT + '/german_coefs.json'):
    print('German already done, skipping.')
else:
    assert run(['credit_benchmark.py', '--data', 'german', '--repeats', '3']) == 0


In [ ]:
# 3) TAIWAN: 2 repeats x 5 folds, 30,000 rows. Use a GPU runtime. Skipped if already finished.
assert 'run' in globals(), 'Run cell 1 first'
if os.path.exists(OUT + '/taiwan_coefs.json'):
    print('Taiwan already done, skipping.')
else:
    assert run(['credit_benchmark.py', '--data', 'taiwan', '--repeats', '2', '--n_max', '30000']) == 0


In [ ]:
# 4) ANALYSIS, FIGURES and VERIFICATION. Writes results.md, fig_reliability.png and fig_paired.png into the out folder.
assert 'run' in globals(), 'Run cell 1 first'
run(['analyze_benchmark.py'])
run(['make_figures.py'])
run(['verify_paper.py'])
print(sorted(os.listdir(OUT)))


Results are in `credit_risk_v2/out`: `results.md`, `fig_reliability.png`, `fig_paired.png` and the CSV files.